In [ ]:
# Importation of necessary libraries
import os
import db_dtypes
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "my-project-ecommerce1-a9cefc290e72.json"
from google.cloud import bigquery
import pandas as pd
import sys
!{sys.executable} -m pip install db-dtypes


In [19]:
# Create a BigQuery client
client = bigquery.Client(project="my-project-ecommerce1")
dataset_ref = client.dataset("ga4_obfuscated_sample_ecommerce", project="bigquery-public-data")
dataset = client.get_dataset(dataset_ref)

In [20]:
# List all tables in the dataset
list_of_tables = [table.table_id for table in client.list_tables(dataset)]
list_of_tables_df = pd.DataFrame(list_of_tables, columns=['table_id'])
print(list_of_tables_df.head())
list_of_tables_df.shape

          table_id
0  events_20201101
1  events_20201102
2  events_20201103
3  events_20201104
4  events_20201105


(92, 1)

In [21]:
# Load the events_20201116 table into a DataFrame to take a look at the data
table_ref = dataset_ref.table('events_20201116')
table = client.get_table(table_ref)
df_events = client.list_rows(table).to_dataframe()
print(df_events.head())


c:\Users\HP\Documents\L3_Math_Info\Projets_Code\bq_ecommerce_project\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


  event_date   event_timestamp     event_name  \
0   20201116  1605500459169424    first_visit   
1   20201116  1605500459169424      page_view   
2   20201116  1605500459169424  session_start   
3   20201116  1605537389044721  session_start   
4   20201116  1605537394063728         scroll   

                                        event_params  \
0  [{'key': 'page_title', 'value': {'string_value...   
1  [{'key': 'page_title', 'value': {'string_value...   
2  [{'key': 'ga_session_id', 'value': {'string_va...   
3  [{'key': 'page_referrer', 'value': {'string_va...   
4  [{'key': 'debug_mode', 'value': {'string_value...   

   event_previous_timestamp  event_value_in_usd  event_bundle_sequence_id  \
0                      <NA>                 NaN                3714052721   
1                      <NA>                 NaN                3714052721   
2                      <NA>                 NaN                3714052721   
3                      <NA>                 NaN             

In [22]:
# Define a function to run a query and return the results as a DataFrame
def run_query(query):
    safe_config = bigquery.QueryJobConfig(maximum_bytes_billed=10**10)
    query_job = client.query(query, job_config=safe_config)
    query_results = query_job.to_dataframe()
    return query_results



In [ ]:
# collect the column names of the DataFrame for checking the features and target variable
df_events.columns

Index(['event_date', 'event_timestamp', 'event_name', 'event_params',
       'event_previous_timestamp', 'event_value_in_usd',
       'event_bundle_sequence_id', 'event_server_timestamp_offset', 'user_id',
       'user_pseudo_id', 'privacy_info', 'user_properties',
       'user_first_touch_timestamp', 'user_ltv', 'device', 'geo', 'app_info',
       'traffic_source', 'stream_id', 'platform', 'event_dimensions',
       'ecommerce', 'items'],
      dtype='str')

In [ ]:
# Create a query to extract features and the target variable from the GA4 e-commerce dataset
query = """
        SELECT user_pseudo_id,
            
            -- 1. Contextual Features (Device, Location, Traffic source)
            MAX(device.category) AS device_category,
            MAX(geo.country) AS country,
            MAX(traffic_source.medium) AS traffic_medium,
            
            -- 2. Behavioral Features (Action counters)
            COUNT(CASE WHEN event_name = 'view_item' THEN 1 END) AS count_view_item,
            COUNT(CASE WHEN event_name = 'add_to_cart' THEN 1 END) AS count_add_to_cart,
            COUNT(CASE WHEN event_name = 'begin_checkout' THEN 1 END) AS count_begin_checkout,
            
            -- 3. TARGET VARIABLE : Did the user make at least one purchase? (1 = Yes, 0 = No)
            MAX(CASE WHEN event_name = 'purchase' THEN 1 ELSE 0 END) AS target_has_purchased

        FROM `bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*`
        WHERE _TABLE_SUFFIX BETWEEN '20201101' AND '20201130'
        AND user_pseudo_id IS NOT NULL
        GROUP BY user_pseudo_id
                """
eco_df = run_query(query)


c:\Users\HP\Documents\L3_Math_Info\Projets_Code\bq_ecommerce_project\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [ ]:
# Display the first few rows of the resulting DataFrame and its shape
print(eco_df.head())
eco_df.shape

       user_pseudo_id device_category        country traffic_medium  \
0  1018325.2675818745         desktop  United States         (none)   
1  1034873.2565456828         desktop  United States        organic   
2  1035102.1465935078          mobile          Spain        <Other>   
3  1070488.0264495067         desktop  United States        organic   
4  1091029.8690027225          mobile    Switzerland       referral   

   count_view_item  count_add_to_cart  count_begin_checkout  \
0                0                  0                     0   
1                0                  0                     0   
2                0                  0                     0   
3                3                  0                     0   
4                0                  0                     0   

   target_has_purchased  
0                     0  
1                     0  
2                     0  
3                     0  
4                     0  


(79421, 8)

In [30]:
# Save the resulting DataFrame to a CSV file
eco_df.to_csv('ecommerce_users_raw.csv', index=False)